In [10]:
"""
CSTR Case Study — Closed-Form Homothetic Tube MPC
Figure: 1x2 layout
  Left:  dCa (top) and dT (bottom) trajectories with contracting tube
  Right: phase portrait showing x0, Z(1), Omega, alpha_0*Omega, trajectory
"""

import numpy as np
import scipy.linalg  as la
import scipy.signal  as sig
import scipy.stats   as stats
import cvxpy         as cp
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch, FancyArrow
from matplotlib.lines   import Line2D
from dataclasses import dataclass
from itertools   import product as iprod
from scipy.optimize import fsolve

FP=8; ONE_COL=3.5; TWO_COL=7.16
plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":         FP,    "axes.labelsize":   FP,
    "axes.titlesize":    FP,    "xtick.labelsize":  FP - 1,
    "ytick.labelsize":   FP - 1,"legend.fontsize":  FP - 1,
    "legend.framealpha": 0.90,  "legend.frameon":   True,
    "legend.edgecolor":  "0.80","lines.linewidth":  1.0,
    "axes.linewidth":    0.6,   "grid.linewidth":   0.4,
    "grid.alpha":        0.40,  "savefig.dpi":      300,
    "savefig.bbox":      "tight","pdf.fonttype":    42,
    "ps.fonttype":       42,
})
C={"p":"#007E5C","r":"#FF4500","ref":"#1D1D1D","constr":"#A80000","x0":"#333399", "constr_in":"#FDDFDF"}

# ── CSTR parameters ────────────────────────────────────────────────────────────
_q=100.;_V=100.;_CAin=1.0;_Tin=350.;_k0=7.2e10;_EaR=8750.
_dH=-5e4;_rho_cp=239.;_UA=5e4
k_r  = lambda T: _k0*np.exp(-_EaR/T)
dk_r = lambda T: k_r(T)*_EaR/T**2
F_ss = lambda x,Tj: [_q/_V*(_CAin-x[0])-k_r(x[1])*x[0],
                     _q/_V*(_Tin-x[1])+(-_dH)/_rho_cp*k_r(x[1])*x[0]
                     -_UA/(_rho_cp*_V)*(x[1]-Tj)]

def Ac_jac(Ca,T):
    return np.array([[-(  _q/_V)-k_r(T),   -dk_r(T)*Ca],
                     [(-_dH/_rho_cp)*k_r(T),
                      -(  _q/_V)-_UA/(_rho_cp*_V)+(-_dH/_rho_cp)*dk_r(T)*Ca]])
Bc_mat = lambda: np.array([[0.],[_UA/(_rho_cp*_V)]])

def discretize(Ca,T,Ts):
    sd=sig.StateSpace(Ac_jac(Ca,T),Bc_mat(),np.eye(2),np.zeros((2,1))).to_discrete(Ts)
    return np.array(sd.A),np.array(sd.B)

def dAd_dEaR(Ca,T,Ts,h=1.0):
    def Ad(EaR):
        kk=_k0*np.exp(-EaR/T); dkk=kk*EaR/T**2
        Ac_=np.array([[-(  _q/_V)-kk,-dkk*Ca],
                      [(-_dH/_rho_cp)*kk,-(_q/_V)-_UA/(_rho_cp*_V)
                       +(-_dH/_rho_cp)*dkk*Ca]])
        sd=sig.StateSpace(Ac_,Bc_mat(),np.eye(2),np.zeros((2,1))).to_discrete(Ts)
        return np.array(sd.A)
    return (Ad(_EaR+h)-Ad(_EaR-h))/(2*h)


# ── Offline helpers ────────────────────────────────────────────────────────────

def _sdp(mats,n):
    Qv=cp.Variable((n,n),symmetric=True)
    cp.Problem(cp.Minimize(cp.trace(Qv)),
               [Qv>>M+1e-9*np.eye(n) for M in mats]
               ).solve(solver="SCS",verbose=False,eps=1e-8,max_iters=15000)
    return 0.5*(Qv.value+Qv.value.T)+1e-9*np.eye(n)

def _box_verts(H,h):
    d=H.shape[1]; half=len(h)//2
    return np.array(list(iprod(*zip(-h[half:],h[:d]))))

def _alpha_seq(alpha_k,c_S,N):
    j=np.arange(N+1,dtype=float)
    return np.sqrt(np.maximum(1.-(1.-c_S)**j*(1.-alpha_k**2),0.))

def _beta(alpha,c_S):
    return float(np.sqrt((1.-c_S)*alpha**2+c_S))


# ── HT-MPC OCP ────────────────────────────────────────────────────────────────

class HTMPC_OCP:
    def __init__(self,A,B,Hx,h_x,Hu,h_u,s_x,s_u,gamma,L_inf,sq_gam,
                 N,Q_c,R_c,P_inf):
        self.N=N; self.s_x=s_x; self.s_u=s_u; self.h_x=h_x; self.h_u=h_u
        n=A.shape[0]; m=B.shape[1]; use_t=gamma>1e-8
        self.z0p=cp.Parameter(n); self.zrp=cp.Parameter(n); self.vrp=cp.Parameter(m)
        self.hzp=[cp.Parameter(Hx.shape[0]) for _ in range(N+1)]
        self.hvp=[cp.Parameter(Hu.shape[0]) for _ in range(N)]
        z=cp.Variable((N+1,n)); v=cp.Variable((N,m))
        self._z=z; self._v=v
        cost=0.; cons=[z[0]==self.z0p]
        for j in range(N):
            cost+=(cp.quad_form(z[j]-self.zrp,np.atleast_2d(Q_c))
                  +cp.quad_form(v[j]-self.vrp,np.atleast_2d(R_c)))
            cons+=[z[j+1]==A@z[j]+B@v[j],
                   Hx@z[j]<=self.hzp[j], Hu@v[j]<=self.hvp[j]]
        cost+=cp.quad_form(z[N]-self.zrp,P_inf)
        cons.append(Hx@z[N]<=self.hzp[N])
        if use_t:
            cons.append(cp.norm(L_inf@(z[N]-self.zrp),2)<=sq_gam)
        self.prob=cp.Problem(cp.Minimize(cost),cons)
        self._slv=cp.CLARABEL if use_t else cp.OSQP

    def solve(self,z_k,z_ref,v_ref,aseq,warm=True):
        self.z0p.value=z_k; self.zrp.value=z_ref; self.vrp.value=v_ref
        for j in range(self.N+1): self.hzp[j].value=self.h_x-aseq[j]*self.s_x
        for j in range(self.N):   self.hvp[j].value=self.h_u-aseq[j]*self.s_u
        try:
            kw=dict(warm_start=warm,verbose=False)
            if self._slv==cp.OSQP:
                self.prob.solve(solver=cp.OSQP,eps_abs=1e-7,eps_rel=1e-7,
                                max_iter=40000,polish=True,**kw)
            else: self.prob.solve(solver=cp.CLARABEL,**kw)
            ok=self.prob.status in("optimal","optimal_inaccurate")
            return (self._z.value.copy(),self._v.value.copy(),True) if ok \
                   else (None,None,False)
        except: return None,None,False


# ── Simulation ─────────────────────────────────────────────────────────────────

@dataclass
class SimResult:
    x: np.ndarray; z: np.ndarray; u: np.ndarray
    alpha: np.ndarray; ok: bool
    viol_x: np.ndarray; viol_u: np.ndarray

def run_htmpc(A,B,A_hat,B_hat,K,Hx,h_x,Hu,h_u,Sigma_w,
              Ps,Ps_inv,c_S,ocp,x0,T,N,rng_seed=None):
    rng=np.random.RandomState(rng_seed)
    n=A.shape[0]; m=B.shape[1]
    Xt=np.zeros((T,n)); Zt=np.zeros((T,n)); Ut=np.zeros((T,m))
    At=np.zeros(T); vx=np.zeros(T,bool); vu=np.zeros(T,bool)
    ok=True; alpha_prev=0.; x_k=x0.copy(); z_k=x0.copy()
    zr=np.zeros(n); vr=np.zeros(m)          # regulate to origin
    for k in range(T):
        e_k=x_k-z_k
        raw=float(np.sqrt(max(float(e_k@Ps_inv@e_k),0.)))
        cap=_beta(alpha_prev,c_S); alpha_k=min(cap,raw); At[k]=alpha_k
        aseq=_alpha_seq(alpha_k,c_S,N)
        z_opt,v_opt,feas=ocp.solve(z_k,zr,vr,aseq)
        if not feas: ok=False; break
        u_k=v_opt[0]+K@e_k
        Xt[k]=x_k; Zt[k]=z_k; Ut[k]=u_k
        vx[k]=bool(np.any(Hx@x_k>h_x+1e-6))
        vu[k]=bool(np.any(Hu@u_k>h_u+1e-6))
        w_k=rng.multivariate_normal(np.zeros(n),Sigma_w)
        x_k=A_hat@x_k+B_hat@u_k+w_k
        z_k=A@Zt[k]+B@v_opt[0]; alpha_prev=alpha_k
    Te=T if ok else k
    return SimResult(x=Xt[:Te],z=Zt[:Te],u=Ut[:Te],alpha=At[:Te],
                     ok=ok,viol_x=vx[:Te],viol_u=vu[:Te])


# ── Figure ─────────────────────────────────────────────────────────────────────

def _ellipse_pts(P,center,scale=1.,n_pts=300):
    scale=max(float(scale),1e-6)
    M=0.5*(P[:2,:2]+P[:2,:2].T)*scale**2+1e-12*np.eye(2)
    L=np.linalg.cholesky(M); th=np.linspace(0.,2.*np.pi,n_pts)
    pts=center[:2,None]+L@np.vstack([np.cos(th),np.sin(th)])
    return pts[0],pts[1]

def make_figure(res,Ps,s_x,h_x,Ts,x0,fname="fig_cstr.png"):
    T_p=len(res.x); ts=np.arange(T_p)*Ts
    s0=float(np.sqrt(Ps[0,0])); s1=float(np.sqrt(Ps[1,1]))
    alp=np.clip(res.alpha,0.,1.)
    Ps_inv=np.linalg.inv(Ps)
    alpha_0=float(np.sqrt(max(float(x0@Ps_inv@x0),0.)))
    alpha_0=min(alpha_0,1.)

    peak_Ca=max(float(np.max(np.abs(res.x[:,0])))*1.30, h_x[0]*0.45)
    peak_T =max(float(np.max(np.abs(res.x[:,1])))*1.30, h_x[1]*0.45)

    # fig=plt.figure(figsize=(ONE_COL, ONE_COL*0.750),dpi=600)
    # ax_ca=fig.add_axes([0.07, 0.55, 0.44, 0.38])
    # ax_T =fig.add_axes([0.07, 0.10, 0.44, 0.38])
    # ax_ph=fig.add_axes([0.60, 0.10, 0.38, 0.83])

    fig=plt.figure(figsize=(ONE_COL, ONE_COL*0.46),dpi=600)
    
    # Adjusted coordinates to give breathing room between left and right sides
    ax_ca=fig.add_axes([0.08, 0.55, 0.40, 0.38]) # Width reduced from 0.44 to 0.40
    ax_T =fig.add_axes([0.08, 0.10, 0.40, 0.38]) # Width reduced from 0.44 to 0.40
    ax_ph=fig.add_axes([0.60, 0.10, 0.35, 0.83]) # Shifted right from 0.60 to 0.62

    # ── Left panels: trajectories ─────────────────────────────────────────────
    for ax,sidx,h,si,ylabel,ylim in [
            (ax_ca,0,h_x[0],s0,r"$\delta C_A$ [mol/L]",peak_Ca),
            (ax_T, 1,h_x[1],s1,r"$\delta T$ [K]",        peak_T)]:
        ax.fill_between(ts,
                        res.z[:,sidx]-alp*si, res.z[:,sidx]+alp*si,
                        color=C["p"],alpha=0.35,lw=0.,zorder=2)
        ax.plot(ts,res.z[:,sidx],color=C["p"],lw=0.5,ls=":",alpha=0.75,zorder=3)
        ax.plot(ts,res.x[:,sidx],color=C["p"],lw=0.8,ls="--",zorder=4)
        ax.axhline(0.,color=C["ref"],lw=0.5,ls="-",alpha=0.45,zorder=1)
        ax.axhline( h,color=C["constr"],lw=0.7,zorder=1)
        ax.axhline(-h,color=C["constr"],lw=0.7,zorder=1)
        ax.scatter([0.],[res.x[0,sidx]],
                   color=C["x0"],s=10,zorder=8,marker="o",clip_on=False)
        ax.set_ylabel(ylabel,labelpad=1)
        ax.set_ylim(-ylim,ylim)
        # ax.set_xlim(left=0.)
        ax.grid(True)
        ax.tick_params(axis="both",pad=1)

    ax_T.set_xlabel(r"Time [min]",labelpad=1)
    ax_ca.set_xticklabels([])
    leg_left=[
        Line2D([0],[0],color=C["p"],lw=0.8,ls="--",label="HT-MPC"),
        # Line2D([0],[0],color=C["p"],lw=0.5,ls=":",alpha=0.75,label="Nominal"),
        Patch(facecolor=C["p"],alpha=0.35,label=r"$\alpha_k\Omega$ tube"),
        Line2D([0],[0],color=C["x0"],marker="o",ms=4,lw=0,label=r"$x_0$"),
        Line2D([0],[0],color=C["constr"],lw=0.7,label="Constraint"),
    ]
    ax_ca.legend(handles=leg_left,fontsize=FP,loc="upper center",
                 bbox_to_anchor=(1,1.375), ncol=4,
                 frameon=False,handlelength=1.2,handletextpad=0.4,
                 labelspacing=0.25,borderpad=0.4, )

    # ── Right panel: phase portrait ───────────────────────────────────────────
    # Story: x0 is outside Z(1) but inside X.
    # Show the two boxes and the HT-MPC trajectory — clean, no ellipses.
    ax=ax_ph
    hC=h_x[0]; hT=h_x[1]
    Z1C=hC-s_x[0]; Z1T=hT-s_x[1]

    # X: outer constraint box (light background)
    # ax.add_patch(plt.Rectangle((-hC,-hT),2*hC,2*hT,
    #                            lw=0.,facecolor="0.96",zorder=0))
    ax.add_patch(plt.Rectangle((-hC,-hT),2*hC,2*hT,
                               lw=0.,facecolor=C["constr_in"],zorder=0))
    ax.add_patch(plt.Rectangle((-hC,-hT),2*hC,2*hT,
                               lw=1.0,edgecolor=C["constr"],facecolor="none",zorder=5))

    # Z(1): tightened inner box — RT-MPC nominal must stay inside this
    ax.add_patch(plt.Rectangle((-Z1C,-Z1T),2*Z1C,2*Z1T,
                               lw=0.9,edgecolor=C["r"],
                               facecolor=C["r"]+"22",ls="--",zorder=1))

    # HT-MPC trajectory converging to origin
    T_show=min(len(res.x),80)
    ax.plot(res.x[:T_show,0],res.x[:T_show,1],
            color=C["p"],lw=0.8,ls="--",alpha=0.90,zorder=6)

    # x0 dot (outside Z(1), inside X — the key visual fact)
    ax.scatter(x0[0],x0[1],color=C["x0"],s=10,zorder=9,marker="o",clip_on=False)

    # origin cross (linearization point / regulation target)
    ax.scatter(0.,0.,color="0.35",s=10,zorder=9,marker="+",linewidths=1.0)

    # Minimal math labels only
    ax.text( hC*0.93, hT*0.94,r"$\mathbb{X}$",
             fontsize=FP+1,color=C["constr"],ha="right",va="top",fontweight="bold")
    ax.text( Z1C*0.82, Z1T*0.88,r"$\mathbb{Z}(1)$",
             fontsize=FP-2,color=C["r"],ha="right",va="top")
    ax.text(x0[0]+0.010, x0[1]+2.4,r"$x_0$",
            fontsize=FP-1,color=C["x0"],ha="right",va="bottom",fontweight="bold")

    # ax.set_xlim(-hC*1.05, hC*1.05)
    # ax.set_ylim(-hT*1.02, hT*1.02)
    ax.set_xlabel(r"$\delta C_A$ [mol/L]",labelpad=0.1)
    ax.set_ylabel(r"$\delta T$ [K]",labelpad=0.1)
    ax.tick_params(axis="both",pad=0.1)
    ax.grid(True)

    # leg_ph=[
    #     Line2D([0],[0],color=C["p"],lw=0.8,ls="--",label="HT-MPC traj."),
    #     Line2D([0],[0],color=C["x0"],marker="o",ms=4,lw=0,label=r"$x_0$"),
    #     Patch(facecolor=C["r"],alpha=0.22,edgecolor=C["r"],lw=0.9,
    #           ls="--",label=r"$\mathbb{Z}(1)$ (RT-MPC)"),
    #     Line2D([0],[0],color=C["constr"],lw=1.0,label=r"$\mathbb{X}$"),
    # ]
    # ax.legend(handles=leg_ph,fontsize=FP-2,loc="lower left",
    #           frameon=True,handlelength=1.2,handletextpad=0.4,
    #           labelspacing=0.25,borderpad=0.4)
    fig.tight_layout()
    fig.savefig(fname,dpi=600,bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved  {fname}")



if __name__=="__main__":

    SEP="="*68
    print(SEP); print("  CSTR  —  HT-MPC vs RT-MPC"); print(SEP)

    # ── Steady states ─────────────────────────────────────────────────────────────────────────────
    Tj1=300.; Tj2=310.
    xs1=np.array(fsolve(F_ss,[0.20,370.],args=(Tj1,))); Ca1,T1=xs1
    xs2=np.array(fsolve(F_ss,[0.10,384.],args=(Tj2,))); Ca2,T2=xs2
    x0=(xs2-xs1).copy()
    print(f"  OP1: Ca={Ca1:.5f}  T={T1:.3f}  Tj={Tj1}")
    print(f"  x0 = {np.round(x0,5)}")

    # ── Discretize ───────────────────────────────────────────────────────────────────────────────
    Ts=0.5; n,m=2,1
    A,B=discretize(Ca1,T1,Ts)

    # ── LQR ─────────────────────────────────────────────────────────────────────────────────────────
    Q_lqr=np.diag([10.,0.01]); R_lqr=np.array([[0.1]])
    P_lqr=la.solve_discrete_are(A,B,Q_lqr,R_lqr)
    K=-la.solve(R_lqr+B.T@P_lqr@B,B.T@P_lqr@A); AK=A+B@K
    rho=float(np.max(np.abs(la.eigvals(AK)))); det=abs(la.det(AK))
    print(f"  rho(AK)={rho:.5f}  |det(AK)|={det:.5f}")
    assert rho<1. and det>1e-9

    # ── Uncertainty ──────────────────────────────────────────────────────────────────────────────────
    sigma_Ea=0.5; sigma_EaR=sigma_Ea/8.314
    D_d=dAd_dEaR(Ca1,T1,Ts); d_flat=D_d.flatten()
    eps_reg=(sigma_EaR*0.01)**2
    Sigma_DA=sigma_EaR**2*np.outer(d_flat,d_flat)+eps_reg*np.eye(n*n)
    Sigma_DA=0.5*(Sigma_DA+Sigma_DA.T)
    Sigma_DB=(sigma_EaR*1e-4)**2*np.eye(n*m)
    Sigma_w=np.diag([5e-7,5e-2])


# ── Constraints and shared bound ──────────────────────────────────────────
h_x=np.array([0.30,50.,0.30,50.]); h_u=np.array([60.,60.])
Hx=np.vstack([np.eye(n),-np.eye(n)]); Hu=np.vstack([np.eye(m),-np.eye(m)])
p=0.95
chi2_DA=float(stats.chi2.ppf(p,df=1))
chi2_DB=float(stats.chi2.ppf(p,df=1))
chi2_w =float(stats.chi2.ppf(p,df=n))
print(f"\n[Offline] p={p} shared bound ...")
verts=list(iprod(*zip(-h_x[n:],h_x[:n])))
def QA(x): L=np.kron(np.eye(n),x.reshape(1,-1)); Q=L@Sigma_DA@L.T; return 0.5*(Q+Q.T)
def QB(u): L=np.kron(np.eye(n),u.reshape(1,-1)); Q=L@Sigma_DB@L.T; return 0.5*(Q+Q.T)
QAm=_sdp([QA(np.array(v)) for v in verts],n)
QBm=_sdp([QB(v) for v in [np.array([60.]),np.array([-60.])]],n)
a=[chi2_DA*np.trace(QAm),chi2_DB*np.trace(QBm),chi2_w*np.trace(Sigma_w)]
s_=sum(np.sqrt(ai) for ai in a); mu=np.array([np.sqrt(ai)/s_ for ai in a])
S_max=(chi2_DA/mu[0])*QAm+(chi2_DB/mu[1])*QBm+(chi2_w/mu[2])*Sigma_w
S_max=0.5*(S_max+S_max.T)

def _stein(lam):
    P=la.solve_discrete_lyapunov(AK/np.sqrt(lam),S_max/(1.-lam)); return 0.5*(P+P.T)
rho2=rho**2; a_,b_=rho2+1e-8,1.-1e-8; gr=(np.sqrt(5.)-1.)/2.
x1_=b_-gr*(b_-a_); f1=np.trace(_stein(x1_))
x2_=a_+gr*(b_-a_); f2=np.trace(_stein(x2_))
for _ in range(400):
    if b_-a_<1e-12: break
    if f1<f2: b_,x2_,f2=x2_,x1_,f1; x1_=b_-gr*(b_-a_); f1=np.trace(_stein(x1_))
    else:     a_,x1_,f1=x1_,x2_,f2; x2_=a_+gr*(b_-a_); f2=np.trace(_stein(x2_))
lam_s=0.5*(a_+b_); Ps=_stein(lam_s); Ps_inv=la.inv(Ps)
Pnh=np.real(la.inv(la.sqrtm(Ps)))
c_S=float(np.clip(float(np.max(np.real(la.eigvals(Pnh@S_max@Pnh))))/(1.-lam_s),0,1.))
s_x=np.array([np.sqrt(float(h@Ps@h)) for h in Hx])
print(f"  c_S={c_S:.6f}  sqrt(P*): Ca={np.sqrt(Ps[0,0]):.5f}  T={np.sqrt(Ps[1,1]):.4f}")
print(f"  s_x={np.round(s_x,5)}  margins={np.round(h_x-s_x,5)}")

# ── Stage/terminal cost ───────────────────────────────────────────────────
Q_c=np.diag([50.,0.1]); R_c=np.array([[0.1]])
P_inf=la.solve_discrete_are(A,B,Q_c,R_c); Pi=la.inv(P_inf)
gammas=[]
for i,h in enumerate(Hx):
    mg=h_x[i]-s_x[i]; sv=float(h@Pi@h)
    if mg>1e-10 and sv>1e-12: gammas.append(mg**2/sv)
for i,h in enumerate(Hu):
    hz=K.T@h; su_i=float(np.sqrt(max(float(hz@Ps@hz),0.)))
    mg=h_u[i]-su_i; sv=float(hz@Pi@hz)
    if mg>1e-10 and sv>1e-12: gammas.append(mg**2/sv)
gamma=float(np.min(gammas)); L_inf=np.linalg.cholesky(P_inf).T
sq_gam=float(np.sqrt(gamma))
s_u=np.array([float(np.sqrt(max(float((K.T@h)@Ps@(K.T@h)),0.))) for h in Hu])
print(f"  gamma={gamma:.5f}  sq_gam={sq_gam:.5f}")

# ── Story A: x0 vs Z(1) check ─────────────────────────────────────────────
in_X  =bool(np.all(Hx@x0<=h_x))
out_Z1=bool(np.any(Hx@x0>h_x-s_x[:4]))
print(f"\n  x0 in X={in_X}   x0 outside Z(1)={out_Z1}")
assert in_X and out_Z1,"Story A conditions not met"

# ── Compile OCP ───────────────────────────────────────────────────────────
N=20
print(f"\n[Compile] N={N} ...")
ocp=HTMPC_OCP(A,B,Hx,h_x,Hu,h_u,s_x,s_u,gamma,L_inf,sq_gam,N,Q_c,R_c,P_inf)

# Verify RT-MPC infeasibility directly
from cvxpy import Variable,Problem,Minimize,quad_form,norm
z_rt=Variable((N+1,n)); v_rt=Variable((N,m))
cost_rt=0.; cons_rt=[z_rt[0]==x0]
hz_fix=h_x-s_x; hv_fix=h_u-s_u
for j in range(N):
    cost_rt+=(quad_form(z_rt[j],np.atleast_2d(Q_c))
             +quad_form(v_rt[j],np.atleast_2d(R_c)))
    cons_rt+=[z_rt[j+1]==A@z_rt[j]+B@v_rt[j],
              Hx@z_rt[j]<=hz_fix, Hu@v_rt[j]<=hv_fix]
cost_rt+=quad_form(z_rt[N],P_inf)
cons_rt+=[Hx@z_rt[N]<=hz_fix,
          norm(L_inf@z_rt[N],2)<=sq_gam]
prob_rt=Problem(Minimize(cost_rt),cons_rt)
prob_rt.solve(solver=cp.CLARABEL,verbose=False)
rt_feas=prob_rt.status in("optimal","optimal_inaccurate")
print(f"  RT-MPC feasible from x0: {rt_feas}  (should be False)")

# ── True plant ────────────────────────────────────────────────────────────
rng_pl=np.random.RandomState(42)
EaR_hat=_EaR+rng_pl.normal(0.,sigma_EaR)
def Ac_pert(Ca,T,EaR):
    kk=_k0*np.exp(-EaR/T); dkk=kk*EaR/T**2
    return np.array([[-(  _q/_V)-kk,-dkk*Ca],
                     [(-_dH/_rho_cp)*kk,-(_q/_V)-_UA/(_rho_cp*_V)+(-_dH/_rho_cp)*dkk*Ca]])
sd_h=sig.StateSpace(Ac_pert(Ca1,T1,EaR_hat),Bc_mat(),np.eye(2),np.zeros((2,1))).to_discrete(Ts)
A_hat=np.array(sd_h.A); B_hat=B.copy()
print(f"\n  EaR_hat={EaR_hat:.4f}  ||DA||_F={la.norm(A_hat-A,'fro'):.6f}")

# ── Simulate ──────────────────────────────────────────────────────────────
T_sim=60
print(f"\n[Simulate] T={T_sim} ({T_sim*Ts:.1f} min) ...")
res=run_htmpc(A,B,A_hat,B_hat,K,Hx,h_x,Hu,h_u,Sigma_w,
              Ps,Ps_inv,c_S,ocp,x0,T_sim,N,rng_seed=7)
print(f"  HT-MPC: ok={res.ok}  state_viol={int(res.viol_x.sum())}"
      f"  input_viol={int(res.viol_u.sum())}")
print(f"  alpha: max={res.alpha.max():.4f}  mean={res.alpha.mean():.4f}"
      f"  final={res.alpha[-1]:.4f}")

# ── Figure ────────────────────────────────────────────────────────────────
print("\n[Figure] ...")
make_figure(res,Ps,s_x,h_x,Ts,x0)
print("\n"+SEP+"\n  Done.\n"+SEP)

  CSTR  —  HT-MPC vs RT-MPC
  OP1: Ca=0.20876  T=369.705  Tj=300.0
  x0 = [-0.10962 14.18268]
  rho(AK)=0.45998  |det(AK)|=0.21158

[Offline] p=0.95 shared bound ...
  c_S=0.874169  sqrt(P*): Ca=0.24674  T=16.7610
  s_x=[ 0.24674 16.76095  0.24674 16.76095]  margins=[ 0.05326 33.23905  0.05326 33.23905]
  gamma=3.23009  sq_gam=1.79725

  x0 in X=True   x0 outside Z(1)=True

[Compile] N=20 ...
  RT-MPC feasible from x0: False  (should be False)

  EaR_hat=8750.0299  ||DA||_F=0.165328

[Simulate] T=60 (30.0 min) ...
  HT-MPC: ok=True  state_viol=0  input_viol=0
  alpha: max=0.0676  mean=0.0263  final=0.0156

[Figure] ...


C:\Users\milad.banitalebi\AppData\Local\Temp\ipykernel_21148\3753892521.py:287: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


  Saved  fig_cstr.png

  Done.
